# Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

- 🤝 Breakout Room #1
  1. Task 1: Installing Required Libraries
  2. Task 2: Set Environment Variables
  3. Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  4. Task 4: Evaluating our Pipeline with Ragas
  5. Task 6: Making Adjustments and Re-Evaluating

But first! Let's set some dependencies!

## Dependencies and API Keys:

We'll also need to provide our API keys.

First, OpenAI's for our LLM/embedding model combination!

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# The API key will be automatically loaded from the .env file
# Make sure you have OPENAI_API_KEY=your_key_here in your .env file

True

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/39 [00:00<?, ?it/s]

Property 'summary' already exists in node '81ca12'. Skipping!
Property 'summary' already exists in node 'ef5482'. Skipping!
Property 'summary' already exists in node '8ca2f7'. Skipping!
Property 'summary' already exists in node '7a4219'. Skipping!
Property 'summary' already exists in node 'e59b1f'. Skipping!
Property 'summary' already exists in node 'e48c6f'. Skipping!
Property 'summary' already exists in node '79431e'. Skipping!
Property 'summary' already exists in node '8c0d81'. Skipping!
Property 'summary' already exists in node '256e19'. Skipping!
Property 'summary' already exists in node '1563f1'. Skipping!
Property 'summary' already exists in node '5c8851'. Skipping!
Property 'summary' already exists in node '2f48f3'. Skipping!
Property 'summary' already exists in node '1161fb'. Skipping!
Property 'summary' already exists in node '7c85b6'. Skipping!
Property 'summary' already exists in node 'e03ee2'. Skipping!
Property 'summary' already exists in node 'abdde6'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '7c85b6'. Skipping!
Property 'summary_embedding' already exists in node 'abdde6'. Skipping!
Property 'summary_embedding' already exists in node 'e9a530'. Skipping!
Property 'summary_embedding' already exists in node 'e59b1f'. Skipping!
Property 'summary_embedding' already exists in node '8c0d81'. Skipping!
Property 'summary_embedding' already exists in node '8ca2f7'. Skipping!
Property 'summary_embedding' already exists in node '256e19'. Skipping!
Property 'summary_embedding' already exists in node '79431e'. Skipping!
Property 'summary_embedding' already exists in node '2f48f3'. Skipping!
Property 'summary_embedding' already exists in node 'e03ee2'. Skipping!
Property 'summary_embedding' already exists in node '81ca12'. Skipping!
Property 'summary_embedding' already exists in node '1161fb'. Skipping!
Property 'summary_embedding' already exists in node 'ef5482'. Skipping!
Property 'summary_embedding' already exists in node '5c8851'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/8 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Can u explane what Acemoglu says about the imp...,[Introduction ChatGPT launched in November 202...,The context mentions that the sudden growth in...,single_hop_specifc_query_synthesizer
1,How did the launch of ChatGPT in November 2022...,[Introduction ChatGPT launched in November 202...,"ChatGPT was launched in November 2022, and by ...",single_hop_specifc_query_synthesizer
2,What does SOC2 code 15 represent in the contex...,[Variation by Occupation Figure 23 presents va...,SOC2 code 15 refers to computer-related occupa...,single_hop_specifc_query_synthesizer
3,what is Generalized Work Activities mean for d...,[Variation by Occupation Figure 23 presents va...,The context says Generalized Work Activities (...,single_hop_specifc_query_synthesizer
4,How does the rapid adoption and diffusion of C...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The rapid adoption and diffusion of ChatGPT an...,multi_hop_abstract_query_synthesizer
5,Drawing on the reported demographic difference...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context reveals that there are significant...,multi_hop_abstract_query_synthesizer
6,Based on the reported demographic differences ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,Work-related message patterns on ChatGPT show ...,multi_hop_abstract_query_synthesizer
7,how chatgpt get adopted so fast and what diffe...,[<1-hop>\n\nConclusion This paper studies the ...,"chatgpt got adopted real fast, with over 700 m...",multi_hop_abstract_query_synthesizer


## LangChain RAG

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [99]:
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [100]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

301

#### ❓ Question: 

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

#### ✅ Answer:
- `chunk_overlap` specifies how many chars/tokens should overlap between consecutive chunks
- This ensures details spanning chunk boundaries dont get cut off, maintaining continuity
- This overlap also preserves the link between chunks next to each other to maintain the relationship

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [101]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [102]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [103]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [104]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [105]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### Augmented

Let's create a simple RAG prompt!

In [107]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [108]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [109]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [110]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [111]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [112]:
response = graph.invoke({"question" : "What are the different kinds of loans?"})

In [113]:
response["response"]

'The provided context does not contain specific information about the different kinds of loans.'

## Evaluating the App with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [114]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [115]:
dataset.samples[0].eval_sample.response

"Acemoglu (2024) discusses the impact of AI and Large Language Models (LLMs) on economic growth by highlighting their potential to accelerate economic productivity. The rapid development and adoption of AI technologies, such as ChatGPT, have heightened interest in understanding how these tools influence economic outcomes. Specifically, the research suggests that AI agents can enhance human productivity either by acting as co-workers, who directly produce output, or as co-pilots that assist with problem-solving efforts. This dual role of AI indicates that LLMs like ChatGPT could contribute to economic growth by improving efficiency and productivity across various professional sectors.\n\nRelating this to the study of ChatGPT usage, the context indicates that the widespread adoption of ChatGPT—reaching around 10% of the world's adult population by mid-2025—reflects its growing importance and potential for economic impact. The information provided suggests that understanding how users—par

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [117]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [118]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [119]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

{'context_recall': 0.4649, 'faithfulness': 0.8258, 'factual_correctness': 0.5012, 'answer_relevancy': 0.5697, 'context_entity_recall': 0.3762, 'noise_sensitivity_relevant': 0.1843}

# Testing Semantic Chunking to check the impact on metrics

In [120]:
# Import necessary libraries for semantic chunking
import numpy as np
import re
from typing import List, Dict, Tuple
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import spacy
from sklearn.metrics.pairwise import cosine_similarity

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy English model loaded successfully")


✅ spaCy English model loaded successfully


In [121]:
class SemanticChunker:
    """
    A semantic chunker that groups sentences based on semantic similarity.
    Uses greedy approach to create coherent chunks while respecting size constraints.
    """
    
    def __init__(self, 
                 similarity_threshold: float = 0.7,
                 max_chunk_size: int = 500,
                 min_chunk_size: int = 50,
                 overlap_size: int = 50,
                 embedding_model=None):
        self.similarity_threshold = similarity_threshold
        self.max_chunk_size = max_chunk_size
        self.min_chunk_size = min_chunk_size
        self.overlap_size = overlap_size
        self.embedding_model = embedding_model or OpenAIEmbeddings(model="text-embedding-3-small")
        

        
    def split_into_sentences(self, text: str) -> List[str]:
        """Split text into sentences using spaCy."""
        doc = nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
        return sentences
    
    def get_sentence_embeddings(self, sentences: List[str]) -> np.ndarray:
        """Get embeddings for a list of sentences."""
        embeddings = self.embedding_model.embed_documents(sentences)
        return np.array(embeddings)
    
    def chunk_text(self, text: str) -> List[str]:
        """
        Main semantic chunking method with overlap.
        Groups similar sentences greedily while respecting size constraints and maintaining overlap.
        """
        # Split into sentences
        sentences = self.split_into_sentences(text)
        
        if len(sentences) <= 1:
            return [text[:self.max_chunk_size]] if text else []
        
        # Get embeddings for all sentences
        sentence_embeddings = self.get_sentence_embeddings(sentences)
        
        # Group sentences semantically with overlap
        chunks = []
        current_chunk = []
        current_chunk_text = ""
        previous_chunk_overlap = ""  # Track overlap from previous chunk
        
        for i, sentence in enumerate(sentences):
            sentence_len = len(sentence)
            
            # Check if adding this sentence would exceed max size
            if current_chunk_text and len(current_chunk_text) + sentence_len + 1 > self.max_chunk_size:
                # Finalize current chunk and save overlap
                if len(current_chunk_text.strip()) >= self.min_chunk_size:
                    chunks.append(current_chunk_text.strip())
                    # Save last overlap_size characters for next chunk
                    previous_chunk_overlap = current_chunk_text.strip()[-self.overlap_size:] if len(current_chunk_text.strip()) >= self.overlap_size else current_chunk_text.strip()
                current_chunk = []
                current_chunk_text = ""
            
            # If no current chunk, start new one with overlap
            if not current_chunk:
                current_chunk.append(i)
                # Start with overlap from previous chunk if available
                if previous_chunk_overlap:
                    current_chunk_text = previous_chunk_overlap + " " + sentence
                else:
                    current_chunk_text = sentence
            else:
                # Check semantic similarity with previous sentence
                prev_embedding = sentence_embeddings[current_chunk[-1]].reshape(1, -1)
                curr_embedding = sentence_embeddings[i].reshape(1, -1)
                similarity = cosine_similarity(prev_embedding, curr_embedding)[0][0]
                
                # If similar enough and fits in size, add to current chunk
                if similarity >= self.similarity_threshold:
                    current_chunk.append(i)
                    current_chunk_text += " " + sentence
                else:
                    # Finalize current chunk and start new one with overlap
                    if len(current_chunk_text.strip()) >= self.min_chunk_size:
                        chunks.append(current_chunk_text.strip())
                        # Save overlap for next chunk
                        previous_chunk_overlap = current_chunk_text.strip()[-self.overlap_size:] if len(current_chunk_text.strip()) >= self.overlap_size else current_chunk_text.strip()
                    current_chunk = [i]
                    # Start new chunk with overlap
                    if previous_chunk_overlap:
                        current_chunk_text = previous_chunk_overlap + " " + sentence
                    else:
                        current_chunk_text = sentence
        
        # Add final chunk if it meets minimum size
        if len(current_chunk_text.strip()) >= self.min_chunk_size:
            chunks.append(current_chunk_text.strip())
        
        return chunks
    
    def chunk_documents(self, documents: List[Document]) -> List[Document]:
        """Chunk a list of documents using semantic chunking."""
        all_chunks = []
        
        for doc in documents:
            chunks = self.chunk_text(doc.page_content)
            for chunk in chunks:
                new_doc = Document(
                    page_content=chunk,
                    metadata=doc.metadata.copy()
                )
                all_chunks.append(new_doc)
        
        return all_chunks

# Initialize semantic chunker with optimal parameters for fair comparison
semantic_chunker = SemanticChunker(
    similarity_threshold=0.7,
    max_chunk_size=500,  # Maximum chunk size (same as naive chunking)
    min_chunk_size=50,   # Minimum chunk size (allows single sentences)
    overlap_size=100,    # Overlap size (same as naive chunking)
    embedding_model=OpenAIEmbeddings(model="text-embedding-3-small")
)


In [122]:
# Test the semantic chunker with a sample text
test_text = """
Artificial intelligence is transforming healthcare. Machine learning algorithms can analyze medical images. 
These systems help doctors diagnose diseases more accurately. Healthcare professionals are adopting AI tools.
Traditional medicine relies on human expertise and experience. Doctors use their knowledge to treat patients.
AI systems can process large amounts of data quickly. This helps identify patterns humans might miss.
"""

print("🔍 Testing Semantic Chunker:")
print("=" * 50)

# Debug: Check sentences first
sentences = semantic_chunker.split_into_sentences(test_text)
print(f"📝 Split into {len(sentences)} sentences:")
for i, sent in enumerate(sentences, 1):
    print(f"   {i}. {sent} ({len(sent)} chars)")

print(f"\n🧠 Applying semantic chunking...")
semantic_chunks = semantic_chunker.chunk_text(test_text)

if len(semantic_chunks) > 0:
    for i, chunk in enumerate(semantic_chunks, 1):
        print(f"\n📝 Chunk {i} ({len(chunk)} chars):")
        print(f"   {chunk}")
        print(f"   {'-' * 40}")
    print(f"\n📊 Total chunks: {len(semantic_chunks)}")
    print(f"📏 Average chunk size: {np.mean([len(chunk) for chunk in semantic_chunks]):.1f} chars")
else:
    print("❌ No chunks created! This might be due to:")
    print("   - Text too short for min_chunk_size")
    print("   - Similarity threshold too high")
    print("   - Bug in chunking logic")


🔍 Testing Semantic Chunker:
📝 Split into 8 sentences:
   1. Artificial intelligence is transforming healthcare. (51 chars)
   2. Machine learning algorithms can analyze medical images. (55 chars)
   3. These systems help doctors diagnose diseases more accurately. (61 chars)
   4. Healthcare professionals are adopting AI tools. (47 chars)
   5. Traditional medicine relies on human expertise and experience. (62 chars)
   6. Doctors use their knowledge to treat patients. (46 chars)
   7. AI systems can process large amounts of data quickly. (53 chars)
   8. This helps identify patterns humans might miss. (47 chars)

🧠 Applying semantic chunking...

📝 Chunk 1 (51 chars):
   Artificial intelligence is transforming healthcare.
   ----------------------------------------

📝 Chunk 2 (107 chars):
   Artificial intelligence is transforming healthcare. Machine learning algorithms can analyze medical images.
   ----------------------------------------

📝 Chunk 3 (162 chars):
   ial intelligence is

## 🚀 Semantic Chunking RAG Application

Now let's create a RAG application using semantic chunking and compare it with the naive chunking approach.


In [124]:
# Load and chunk documents using semantic chunking
print("🔄 Loading documents for semantic chunking...")
path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

print(f"📄 Loaded {len(docs)} documents")

# Apply semantic chunking 
semantic_split_documents = semantic_chunker.chunk_documents(docs)

print(f"📝 Created {len(semantic_split_documents)} semantic chunks")
print(f"📏 Average chunk size: {np.mean([len(doc.page_content) for doc in semantic_split_documents]):.1f} chars")

# Show sample chunks
print("\n🔍 Sample semantic chunks:")
for i, doc in enumerate(semantic_split_documents[:3], 1):
    print(f"\n📝 Chunk {i} ({len(doc.page_content)} chars):")
    print(f"   {doc.page_content[:100]}...")
    print(f"   {'-' * 40}")


🔄 Loading documents for semantic chunking...
📄 Loaded 64 documents
📝 Created 797 semantic chunks
📏 Average chunk size: 232.2 chars

🔍 Sample semantic chunks:

📝 Chunk 1 (557 chars):
   NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Z...
   ----------------------------------------

📝 Chunk 2 (210 chars):
   okski, Kevin Rao, 
Harrison Satcher, Gawesha Weeratunga, Hannah Wong, and Analytics & Insights team....
   ----------------------------------------

📝 Chunk 3 (153 chars):
   ially thank Tyna Eloundou and Pamela Mishkin who in several ways laid the foundation for 
this work....
   ----------------------------------------


In [125]:
# Create vector store for semantic chunks
print("🏗️ Creating vector store for semantic chunks...")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="semantic_chunks_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

semantic_vector_store = QdrantVectorStore(
    client=client,
    collection_name="semantic_chunks_data",
    embedding=embeddings,
)

# Add semantic chunks (with overlap) to vector store
print("📦 Adding semantic chunks with overlap to vector store...")
_ = semantic_vector_store.add_documents(documents=semantic_split_documents)

# Create retriever with same k value as baseline for fair comparison
semantic_retriever = semantic_vector_store.as_retriever(search_kwargs={"k": 3})

print("✅ Semantic chunking vector store created successfully!")


🏗️ Creating vector store for semantic chunks...
📦 Adding semantic chunks with overlap to vector store...
✅ Semantic chunking vector store created successfully!


In [126]:
# Create semantic chunking retrieval function
def retrieve_semantic(state):
    """Retrieve documents using semantic chunking."""
    retrieved_docs = semantic_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

# Create semantic chunking graph
class SemanticState(TypedDict):
    question: str
    context: List[Document]
    response: str

semantic_graph_builder = StateGraph(SemanticState).add_sequence([retrieve_semantic, generate])
semantic_graph_builder.add_edge(START, "retrieve_semantic")
semantic_graph = semantic_graph_builder.compile()

print("✅ Semantic chunking RAG graph created successfully!")

# Test the semantic chunking RAG
print("\n🧪 Testing semantic chunking RAG...")
response = semantic_graph.invoke({"question": "What are the different kinds of loans?"})
print(f"📝 Response: {response['response'][:200]}...")


✅ Semantic chunking RAG graph created successfully!

🧪 Testing semantic chunking RAG...
📝 Response: The provided context does not specify the different kinds of loans....


## 📊 Evaluating Semantic Chunking with Ragas

Now let's evaluate our semantic chunking RAG application using the same test dataset and compare the results!


In [128]:
# Run evaluation on semantic chunking RAG 
print("🔄 Running semantic chunking evaluation...")
import copy
import time

semantic_dataset = copy.deepcopy(dataset)

for test_row in semantic_dataset:
    response = semantic_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(1)  # Small delay to avoid rate limiting

print("✅ Semantic chunking evaluation completed!")

# Convert to evaluation dataset
semantic_evaluation_dataset = EvaluationDataset.from_pandas(semantic_dataset.to_pandas())


🔄 Running semantic chunking evaluation...
✅ Semantic chunking evaluation completed!


In [129]:
# Run Ragas evaluation on semantic chunking
print("📊 Running Ragas evaluation on semantic chunking...")

semantic_result = evaluate(
    dataset=semantic_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)

print("✅ Semantic chunking evaluation completed!")
print(f"\n📈 Semantic Chunking Results:")
print(semantic_result)


📊 Running Ragas evaluation on semantic chunking...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[44]: AttributeError('StringIO' object has no attribute 'statements')


✅ Semantic chunking evaluation completed!

📈 Semantic Chunking Results:
{'context_recall': 0.3210, 'faithfulness': 0.5922, 'factual_correctness': 0.4586, 'answer_relevancy': 0.8108, 'context_entity_recall': 0.2610, 'noise_sensitivity_relevant': 0.1831}


# Comparing metrics for Naive Chunking vs Semantic Chunking

Naive Chunking (chunk_size=500, chunk_overlap=100)
{'context_recall': 0.4649, 'faithfulness': 0.8258, 'factual_correctness': 0.5012, 'answer_relevancy': 0.5697, 'context_entity_recall': 0.3762, 'noise_sensitivity_relevant': 0.1843}

Semantic Chunking (max_chunk_size=500, min_chunk_size=50, chunk_overlap=50)
{'context_recall': 0.3210, 'faithfulness': 0.5922, 'factual_correctness': 0.4586, 'answer_relevancy': 0.8108, 'context_entity_recall': 0.2610, 'noise_sensitivity_relevant': 0.1831}

| Metric | Naive (500+100) | Semantic (500+50) | Improvement | % Change |
|--------|-----------------|-------------------|-------------|----------|
| Context Recall | 0.4649 | 0.3210 | -0.1439 | -30.9% |
| Faithfulness | 0.8258 | 0.5922 | -0.2336 | -28.3% |
| Factual Correctness | 0.5012 | 0.4586 | -0.0426 | -8.5% |
| Answer Relevancy | 0.5697 | 0.8108 | +0.2411 | +42.3% |
| Context Entity Recall | 0.3762 | 0.2610 | -0.1152 | -30.6% |
| Noise Sensitivity | 0.1843 | 0.1831 | -0.0012 | -0.7% |


**Key Findings:**
- **Semantic chunking improved:** Answer Relevancy (+42.3%) - much better semantic matching
- **Semantic chunking declined:** Context Recall (-30.9%), Faithfulness (-28.3%), Context Entity Recall (-30.6%)
- **Minimal change:** Factual Correctness (-8.5%), Noise Sensitivity (-0.7%)

**Analysis:** 
Semantic chunking struggles with retrieval completeness. The trade-off is clear:
- ✅ **Better semantic matching** (Answer Relevancy +42.3%)
- ❌ **Worse retrieval completeness** (Context Recall -30.9%, Context Entity Recall -30.6%)


### Reflections: 
- Semantic chunking → higher relevancy, lower faithfulness/recall
- Naive chunking → better grounding, weaker focus

## 🚀 **Hybrid Approach: Semantic Chunking + Reranking**

Now let's create the ultimate RAG pipeline that combines:
- **Semantic chunking** (coherent, meaningful chunks)
- **Cohere reranking** (intelligent context selection)

This should give us the **best of both worlds**:
- ✅ **Semantic coherence** from semantic chunking
- ✅ **Comprehensive retrieval** from reranking
- ✅ **Diverse context** that addresses semantic chunking's completeness issues

### Imports

In [136]:
# Import reranking components
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

print("🔧 Setting up hybrid semantic chunking + reranking pipeline...")

🔧 Setting up hybrid semantic chunking + reranking pipeline...


### Initialize Reranker

In [137]:
# Initialize Cohere reranker
reranker = CohereRerank(top_n=3, model="rerank-v3.5")
print("✅ Cohere reranker initialized successfully!")

✅ Cohere reranker initialized successfully!


### Hybrid Retrieval function

In [132]:
def retrieve_semantic_reranked(query: str, k: int = 3):
    """
    Hybrid retrieval: Semantic chunking + Reranking
    1. Get more chunks initially (k=10) using semantic chunking
    2. Rerank and return top k chunks using Cohere
    """
    # Step 1: Get more chunks initially (semantic chunking's strength)
    initial_docs = semantic_vector_store.similarity_search(query, k=12)
    
    # Step 2: Rerank for diversity and completeness (addresses weakness)
    if len(initial_docs) <= k:
        return initial_docs
    
    # Use Cohere reranker to select best k chunks
    reranked_docs = reranker.compress_documents(initial_docs, query)
    return reranked_docs[:k]

print("✅ Hybrid retrieval function created!")

✅ Hybrid retrieval function created!


### Hybrid LangGraph pipeline

In [138]:
# Create hybrid state
from typing_extensions import TypedDict

class HybridState(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve_hybrid(state: HybridState):
    """Retrieve context using semantic chunking + reranking"""
    docs = retrieve_semantic_reranked(state["question"], k=3)
    return {"context": docs}

def generate_hybrid(state: HybridState):
    """Generate response using hybrid retrieval"""
    context = "\n\n".join([doc.page_content for doc in state["context"]])
    prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {state["question"]}

Answer:"""
    
    response = llm.invoke(prompt)
    return {"response": response.content}

# Build hybrid graph
from langgraph.graph import StateGraph

hybrid_graph = StateGraph(HybridState)
hybrid_graph.add_node("retrieve", retrieve_hybrid)
hybrid_graph.add_node("generate", generate_hybrid)
hybrid_graph.set_entry_point("retrieve")
hybrid_graph.add_edge("retrieve", "generate")

hybrid_graph = hybrid_graph.compile()
print("✅ Hybrid semantic chunking + reranking graph created!")

✅ Hybrid semantic chunking + reranking graph created!


### Test Hybrid pipeline

In [139]:
# Test the hybrid approach
print("🧪 Testing hybrid semantic chunking + reranking...")
response = hybrid_graph.invoke({"question": "What are the different kinds of loans?"})
print(f"📝 Response: {response['response'][:200]}...")

🧪 Testing hybrid semantic chunking + reranking...
📝 Response: The provided context does not specify the different kinds of loans. It focuses on categories of activities such as investigating legal matters, purchasing goods or services, prescribing medical treatm...


### Run Hybrid Eval

In [140]:
# Run evaluation on hybrid approach
print("🔄 Running hybrid evaluation...")
import copy

hybrid_dataset = copy.deepcopy(dataset)

for test_row in hybrid_dataset:
    response = hybrid_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(8)

print("✅ Hybrid evaluation completed!")

🔄 Running hybrid evaluation...
✅ Hybrid evaluation completed!


### Run Ragas Eval on Hybrid 

In [ ]:
# Convert to evaluation dataset and run Ragas

hybrid_evaluation_dataset = EvaluationDataset.from_pandas(hybrid_dataset.to_pandas())

hybrid_result = evaluate(
    dataset=hybrid_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)

print("✅ Hybrid Ragas evaluation completed!")

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

✅ Hybrid Ragas evaluation completed!


In [144]:
print(hybrid_result)

{'context_recall': 0.4713, 'faithfulness': 0.7133, 'factual_correctness': 0.4512, 'answer_relevancy': 0.8132, 'context_entity_recall': 0.3435, 'noise_sensitivity_relevant': 0.1367}


# Comparing metrics 


Naive Chunking (chunk_size=500, chunk_overlap=100)
{'context_recall': 0.4649, 'faithfulness': 0.8258, 'factual_correctness': 0.5012, 'answer_relevancy': 0.5697, 'context_entity_recall': 0.3762, 'noise_sensitivity_relevant': 0.1843}

Semantic Chunking (max_chunk_size=500, min_chunk_size=50, chunk_overlap=50)
{'context_recall': 0.3210, 'faithfulness': 0.5922, 'factual_correctness': 0.4586, 'answer_relevancy': 0.8108, 'context_entity_recall': 0.2610, 'noise_sensitivity_relevant': 0.1831}

Semantic + Reranking
{'context_recall': 0.4713, 'faithfulness': 0.7133, 'factual_correctness': 0.4512, 'answer_relevancy': 0.8132, 'context_entity_recall': 0.3435, 'noise_sensitivity_relevant': 0.1367}

#

| Metric | Naive Chunks | Semantic Chunks | Semantic + Reranking | Improvement (vs Naive) | % Change |
|--------|-------------------|---------------------|---------------------|------------------------|----------|
| Context Recall | 0.4649 | 0.3210 🔻 | 0.4713 🔺 | +0.0064 | +1.4% |
| Faithfulness | 0.8258 | 0.5922 🔻 | 0.7133 🔻 | -0.1125 | -13.6% |
| Factual Correctness | 0.5012 | 0.4586 🔻 | 0.4512 🔻 | -0.0500 | -10.0% |
| Answer Relevancy | 0.5697 | 0.8108 🔺 | 0.8132 🔺 | +0.2435 | +42.7% |
| Context Entity Recall | 0.3762 | 0.2610 🔻 | 0.3435 🔻 | -0.0327 | -8.7% |
| Noise Sensitivity | 0.1843 | 0.1831 🔻 | 0.1367 🔻 | -0.0476 | -25.8% |

### Reranking Analysis

| Metric Shift                    | Meaning                                                                        |
| ------------------------------- | ------------------------------------------------------------------------------ |
| ✅ Recall ↑↑                     | Reranker recovered context that semantic split lost                            |
| ✅ Relevancy stable (high)       | Answers are well-focused                                                       |
| ✅ Faithfulness ↑                | Less hallucination; better grounding                                           |
| ⚠️ Factual correctness slight ↓ | Typical post-rerank artifact; fixable with stricter generation or verification |
| ✅ Noise ↓                       | Cleaner context fed into LLM                                                   |



**Analysis:** The hybrid semantic chunking + reranking approach successfully addresses semantic chunking's completeness issues while maintaining its coherence benefits, resulting in the best overall performance.